# ECSC Influence Analysis: Residual Influence vs Retrieval

Tests whether predictive support is:
1. **Continuous** beyond STM range (not just recent tokens)
2. **Not bursty/sparse** (influence spread across bins, not concentrated)

Method: Leave-one-bin-out ablation to measure each context bin's contribution.

## Cell 1: Setup and Configuration

In [ ]:
# ============================================================
# CONFIGURATION - EDIT THESE
# ============================================================

# Google Drive paths
DRIVE_BASE = "/content/drive/MyDrive/LRTIA"
DATA_FILE = f"{DRIVE_BASE}/Data/ecsc_processed/concatenated_docs.jsonl"
OUTPUT_DIR = f"{DRIVE_BASE}/results/ecsc_influence_v1"

# Model settings
MODEL_NAME = "mistralai/Mistral-7B-v0.1"
USE_4BIT = True  # Set False for full precision (needs more VRAM)

# Analysis parameters
MAX_CONTEXT = 384       # Maximum context length to analyze
BIN_SIZE = 32           # Size of each ablation bin
TARGET_LEN = 30         # Tokens to score in target region
BURN_IN = 64            # Tokens before target (always included as close context)
SLIDING_WINDOW = 16     # Size of sliding ablation window for continuous analysis

# Processing
MAX_DOCS = None         # Set to int for testing (e.g., 20)
MIN_TOKENS = 200        # Minimum tokens required for analysis

print(f"Config:")
print(f"  MAX_CONTEXT: {MAX_CONTEXT}")
print(f"  BIN_SIZE: {BIN_SIZE}")
print(f"  N_BINS: {MAX_CONTEXT // BIN_SIZE}")
print(f"  TARGET_LEN: {TARGET_LEN}")
print(f"  BURN_IN: {BURN_IN}")

## Cell 2: Install Dependencies and Mount Drive

In [ ]:
# Install packages
!pip install -q transformers accelerate bitsandbytes scipy

# Mount Google Drive
import os
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print("Drive already mounted")

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

# Check data file
if os.path.exists(DATA_FILE):
    print(f"Data file found: {DATA_FILE}")
else:
    print(f"ERROR: Data file not found: {DATA_FILE}")
    print("Available files:")
    !ls -la {DRIVE_BASE}/Data/ 2>/dev/null || echo "Directory not found"

## Cell 3: Load Model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print(f"Loading model: {MODEL_NAME}")
print(f"4-bit quantization: {USE_4BIT}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto",
    )

model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Cell 4: Load Data

In [ ]:
import json
import pandas as pd

# Load documents
docs = []
with open(DATA_FILE, 'r') as f:
    for line in f:
        if line.strip():
            doc = json.loads(line)
            pop = json.loads(doc['population'])
            doc['age_months'] = pop.get('age_months')
            doc['subject_id'] = pop.get('subject_id')
            docs.append(doc)

print(f"Loaded {len(docs)} documents")

# Tokenize and filter by length
for doc in docs:
    doc['tokens'] = tokenizer.encode(doc['text'], add_special_tokens=False)
    doc['n_tokens'] = len(doc['tokens'])

docs = [d for d in docs if d['n_tokens'] >= MIN_TOKENS]
print(f"After length filter (>= {MIN_TOKENS} tokens): {len(docs)} documents")

if MAX_DOCS:
    docs = docs[:MAX_DOCS]
    print(f"Limited to {MAX_DOCS} documents for testing")

# Age distribution
ages = [d['age_months'] for d in docs if d.get('age_months')]
print(f"\nAge range: {min(ages)}-{max(ages)} months")

## Cell 5: Core Functions - NLL Computation

In [ ]:
import numpy as np

@torch.no_grad()
def compute_nll_on_target(token_ids, target_start, target_end):
    """
    Compute mean NLL on target region [target_start, target_end).
    Context is tokens [0:target_start].
    """
    if target_end > len(token_ids):
        target_end = len(token_ids)
    if target_start >= target_end - 1:
        return float('inf')
    
    input_ids = torch.tensor([token_ids], device=device)
    outputs = model(input_ids)
    logits = outputs.logits[0]  # [seq_len, vocab_size]
    
    total_nll = 0.0
    count = 0
    
    for i in range(target_start, target_end - 1):
        log_probs = torch.log_softmax(logits[i], dim=-1)
        target_token = token_ids[i + 1]
        token_nll = -log_probs[target_token].item()
        total_nll += token_nll
        count += 1
    
    return total_nll / count if count > 0 else float('inf')


def ablate_bin(tokens, bin_start, bin_end):
    """
    Create token sequence with bin [bin_start:bin_end] removed.
    Returns: (ablated_tokens, new_target_start, new_target_end)
    """
    ablated = tokens[:bin_start] + tokens[bin_end:]
    removed_len = bin_end - bin_start
    return ablated, removed_len


print("NLL computation functions defined.")

## Cell 6: Bin-Ablation Influence Analysis

In [ ]:
from tqdm import tqdm

def compute_bin_influences(doc, max_context, bin_size, burn_in, target_len):
    """
    Compute leave-one-bin-out influence for each context bin.
    
    Layout:
    [distant context bins][burn_in][target region]
    
    We ablate bins in the "distant context" area (before burn_in).
    """
    tokens = doc['tokens']
    n_tokens = len(tokens)
    
    # Define regions
    # Target starts after max_context, but we need at least burn_in before it
    # So effective context = max_context tokens before target
    
    # For this analysis:
    # - target_start = position where we start scoring
    # - context_end = target_start (tokens before this are context)
    # - context_start = max(0, target_start - max_context)
    
    # We need: context + target to fit in available tokens
    total_needed = max_context + target_len
    if n_tokens < total_needed:
        # Adjust for shorter docs
        available_context = n_tokens - target_len - burn_in
        if available_context < bin_size:
            return None  # Too short
        actual_context = min(available_context, max_context - burn_in)
    else:
        actual_context = max_context - burn_in
    
    # Position the window
    target_end_pos = min(n_tokens, max_context + target_len)
    target_start_pos = target_end_pos - target_len
    context_start_pos = target_start_pos - burn_in - actual_context
    
    if context_start_pos < 0:
        context_start_pos = 0
        actual_context = target_start_pos - burn_in
    
    # Extract the working sequence
    work_tokens = tokens[context_start_pos:target_end_pos]
    work_len = len(work_tokens)
    
    # In work_tokens:
    # [0 : actual_context] = distant context (ablatable)
    # [actual_context : actual_context + burn_in] = burn-in (always kept)
    # [actual_context + burn_in : end] = target region
    
    local_target_start = actual_context + burn_in
    local_target_end = work_len
    
    # Define bins in the distant context region
    n_bins = actual_context // bin_size
    if n_bins == 0:
        return None
    
    # Compute baseline NLL (full context)
    baseline_nll = compute_nll_on_target(work_tokens, local_target_start, local_target_end)
    if baseline_nll == float('inf'):
        return None
    
    # Compute influence for each bin
    bin_influences = []
    
    for i in range(n_bins):
        bin_start = i * bin_size
        bin_end = (i + 1) * bin_size
        
        # Distance from target (in tokens)
        bin_center = (bin_start + bin_end) / 2
        distance_from_target = local_target_start - bin_center
        
        # Ablate this bin
        ablated_tokens = work_tokens[:bin_start] + work_tokens[bin_end:]
        new_target_start = local_target_start - bin_size
        new_target_end = local_target_end - bin_size
        
        # Compute NLL with bin ablated
        ablated_nll = compute_nll_on_target(ablated_tokens, new_target_start, new_target_end)
        
        # Influence = increase in NLL when bin is removed
        # Positive = bin was helpful
        delta_nll = ablated_nll - baseline_nll
        
        bin_influences.append({
            'bin_idx': i,
            'bin_start': bin_start,
            'bin_end': bin_end,
            'distance': distance_from_target,
            'delta_nll': delta_nll,
            'ablated_nll': ablated_nll,
        })
    
    return {
        'baseline_nll': baseline_nll,
        'n_bins': n_bins,
        'actual_context': actual_context,
        'bin_influences': bin_influences,
    }


print("Bin-ablation function defined.")

## Cell 7: Compute Metrics from Bin Influences

In [ ]:
def gini_coefficient(values):
    """Compute Gini coefficient (0 = equal, 1 = concentrated)."""
    values = np.array(values)
    values = np.maximum(values, 0)  # Treat negative as 0
    if values.sum() == 0:
        return 0
    values = np.sort(values)
    n = len(values)
    cumsum = np.cumsum(values)
    return (2 * np.sum((np.arange(1, n+1) * values)) / (n * cumsum[-1])) - (n + 1) / n


def compute_influence_metrics(result, stm_threshold=128):
    """
    Compute tail_influence_share and burstiness metrics.
    
    Args:
        result: Output from compute_bin_influences
        stm_threshold: Distance threshold for "tail" (beyond STM range)
    """
    if result is None:
        return None
    
    bins = result['bin_influences']
    deltas = np.array([b['delta_nll'] for b in bins])
    distances = np.array([b['distance'] for b in bins])
    
    # Only consider positive influences (bins that help prediction)
    positive_deltas = np.maximum(deltas, 0)
    total_influence = positive_deltas.sum()
    
    if total_influence == 0:
        return None
    
    # Tail influence: bins beyond stm_threshold
    tail_mask = distances > stm_threshold
    tail_influence = positive_deltas[tail_mask].sum()
    tail_share = tail_influence / total_influence
    
    # Burstiness metrics
    gini = gini_coefficient(positive_deltas)
    
    # Top-1 bin share
    top1_share = positive_deltas.max() / total_influence
    
    # Top-10% bins share
    n_top = max(1, len(positive_deltas) // 10)
    sorted_deltas = np.sort(positive_deltas)[::-1]
    top10_share = sorted_deltas[:n_top].sum() / total_influence
    
    return {
        'tail_influence_share': tail_share,
        'gini': gini,
        'top1_share': top1_share,
        'top10_share': top10_share,
        'total_influence': total_influence,
        'n_positive_bins': (positive_deltas > 0).sum(),
        'mean_delta': deltas.mean(),
        'max_delta': deltas.max(),
        'n_bins': len(bins),
    }


print("Metrics functions defined.")

## Cell 8: Run Bin-Ablation Analysis

In [ ]:
print(f"Running bin-ablation analysis on {len(docs)} documents...")
print(f"Settings: max_context={MAX_CONTEXT}, bin_size={BIN_SIZE}, burn_in={BURN_IN}")

all_results = []
all_bin_data = []  # For distance profile

for doc in tqdm(docs, desc="Processing"):
    result = compute_bin_influences(
        doc, 
        max_context=MAX_CONTEXT,
        bin_size=BIN_SIZE,
        burn_in=BURN_IN,
        target_len=TARGET_LEN
    )
    
    if result is None:
        continue
    
    metrics = compute_influence_metrics(result, stm_threshold=128)
    if metrics is None:
        continue
    
    # Store document-level results
    row = {
        'doc_id': doc['doc_id'],
        'age_months': doc.get('age_months'),
        'n_tokens': doc['n_tokens'],
        'baseline_nll': result['baseline_nll'],
        **metrics
    }
    all_results.append(row)
    
    # Store bin-level data for distance profile
    for b in result['bin_influences']:
        all_bin_data.append({
            'doc_id': doc['doc_id'],
            'age_months': doc.get('age_months'),
            **b
        })

print(f"\nCompleted: {len(all_results)} documents with valid results")

# Convert to DataFrames
results_df = pd.DataFrame(all_results)
bin_df = pd.DataFrame(all_bin_data)

print(f"Total bin observations: {len(bin_df)}")

## Cell 9: Summary Statistics

In [ ]:
print("=" * 60)
print("INFLUENCE ANALYSIS SUMMARY")
print("=" * 60)

# Overall statistics
print(f"\nN documents: {len(results_df)}")
print(f"\nTail Influence Share (bins > 128 tokens from target):")
print(f"  Mean: {results_df['tail_influence_share'].mean():.3f}")
print(f"  Std:  {results_df['tail_influence_share'].std():.3f}")
print(f"  95% CI: [{results_df['tail_influence_share'].mean() - 1.96*results_df['tail_influence_share'].sem():.3f}, "
      f"{results_df['tail_influence_share'].mean() + 1.96*results_df['tail_influence_share'].sem():.3f}]")

print(f"\nBurstiness Metrics:")
for metric in ['gini', 'top1_share', 'top10_share']:
    mean = results_df[metric].mean()
    sem = results_df[metric].sem()
    print(f"  {metric}: {mean:.3f} [{mean-1.96*sem:.3f}, {mean+1.96*sem:.3f}]")

# By age group
print(f"\n" + "=" * 60)
print("BY AGE GROUP")
print("=" * 60)

def age_bin(months):
    if months is None:
        return 'unknown'
    years = months / 12
    if years < 6:
        return '5yr'
    elif years < 7:
        return '6yr'
    elif years < 8:
        return '7yr'
    elif years < 9:
        return '8yr'
    elif years < 10:
        return '9yr'
    else:
        return '10+yr'

results_df['age_group'] = results_df['age_months'].apply(age_bin)

age_summary = results_df.groupby('age_group').agg({
    'tail_influence_share': ['mean', 'std', 'count'],
    'gini': 'mean',
    'top1_share': 'mean',
}).round(3)
print(age_summary)

## Cell 10: Distance Profile Plot

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: Distance Profile (ΔNLL vs distance)
ax1 = axes[0]

# Group by distance bins for cleaner plot
bin_df['dist_bin'] = (bin_df['distance'] // BIN_SIZE) * BIN_SIZE
dist_profile = bin_df.groupby('dist_bin')['delta_nll'].agg(['mean', 'std', 'count'])
dist_profile['sem'] = dist_profile['std'] / np.sqrt(dist_profile['count'])

ax1.errorbar(dist_profile.index, dist_profile['mean'], 
             yerr=1.96*dist_profile['sem'], 
             marker='o', capsize=3, linewidth=2)
ax1.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax1.axvline(128, color='red', linestyle='--', alpha=0.5, label='STM threshold (128)')
ax1.set_xlabel('Distance from Target (tokens)', fontsize=12)
ax1.set_ylabel('Mean ΔNLL (influence)', fontsize=12)
ax1.set_title('Influence vs Distance', fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Histogram of bin contributions
ax2 = axes[1]
ax2.hist(bin_df['delta_nll'], bins=50, edgecolor='black', alpha=0.7)
ax2.axvline(0, color='red', linestyle='--', alpha=0.7)
ax2.set_xlabel('ΔNLL (bin influence)', fontsize=12)
ax2.set_ylabel('Frequency', fontsize=12)
ax2.set_title('Distribution of Bin Influences', fontsize=14)
ax2.grid(True, alpha=0.3)

# Plot 3: Lorenz curve (concentration)
ax3 = axes[2]

# Compute Lorenz curve from all positive bin influences
positive_influences = bin_df[bin_df['delta_nll'] > 0]['delta_nll'].values
sorted_inf = np.sort(positive_influences)
cumsum = np.cumsum(sorted_inf)
cumsum_norm = cumsum / cumsum[-1]
x_lorenz = np.linspace(0, 1, len(cumsum_norm))

ax3.plot(x_lorenz, cumsum_norm, 'b-', linewidth=2, label='Observed')
ax3.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect equality')
ax3.fill_between(x_lorenz, x_lorenz, cumsum_norm, alpha=0.2)
ax3.set_xlabel('Cumulative share of bins', fontsize=12)
ax3.set_ylabel('Cumulative share of influence', fontsize=12)
ax3.set_title(f'Lorenz Curve (Gini={results_df["gini"].mean():.3f})', fontsize=14)
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/influence_analysis_plots.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_DIR}/influence_analysis_plots.png")

## Cell 11: Sliding Window Influence Density

In [ ]:
def compute_sliding_influence(doc, max_context, window_size, burn_in, target_len, stride=8):
    """
    Compute continuous influence density by sliding a small ablation window.
    Returns influence at each position.
    """
    tokens = doc['tokens']
    n_tokens = len(tokens)
    
    # Same layout as bin analysis
    total_needed = max_context + target_len
    if n_tokens < total_needed:
        actual_context = n_tokens - target_len - burn_in - window_size
        if actual_context < window_size:
            return None
    else:
        actual_context = max_context - burn_in
    
    target_end_pos = min(n_tokens, max_context + target_len)
    target_start_pos = target_end_pos - target_len
    context_start_pos = target_start_pos - burn_in - actual_context
    
    if context_start_pos < 0:
        context_start_pos = 0
        actual_context = target_start_pos - burn_in
    
    work_tokens = tokens[context_start_pos:target_end_pos]
    work_len = len(work_tokens)
    local_target_start = actual_context + burn_in
    local_target_end = work_len
    
    # Baseline
    baseline_nll = compute_nll_on_target(work_tokens, local_target_start, local_target_end)
    if baseline_nll == float('inf'):
        return None
    
    # Slide window across distant context
    influences = []
    
    for pos in range(0, actual_context - window_size + 1, stride):
        win_start = pos
        win_end = pos + window_size
        
        distance = local_target_start - (win_start + win_end) / 2
        
        # Ablate window
        ablated = work_tokens[:win_start] + work_tokens[win_end:]
        new_target_start = local_target_start - window_size
        new_target_end = local_target_end - window_size
        
        ablated_nll = compute_nll_on_target(ablated, new_target_start, new_target_end)
        delta = ablated_nll - baseline_nll
        
        influences.append({
            'position': pos,
            'distance': distance,
            'delta_nll': delta,
        })
    
    return {
        'baseline_nll': baseline_nll,
        'influences': influences,
    }


print(f"Computing sliding-window influence (window={SLIDING_WINDOW})...")
print("This may take a while...")

# Run on subset for speed
sliding_docs = docs[:min(50, len(docs))]
sliding_results = []

for doc in tqdm(sliding_docs, desc="Sliding window"):
    result = compute_sliding_influence(
        doc,
        max_context=MAX_CONTEXT,
        window_size=SLIDING_WINDOW,
        burn_in=BURN_IN,
        target_len=TARGET_LEN,
        stride=8
    )
    if result:
        for inf in result['influences']:
            inf['doc_id'] = doc['doc_id']
            sliding_results.append(inf)

sliding_df = pd.DataFrame(sliding_results)
print(f"Sliding window observations: {len(sliding_df)}")

## Cell 12: Continuous Influence Curve Plot

In [ ]:
# Plot continuous influence vs distance
fig, ax = plt.subplots(figsize=(10, 5))

# Group by distance
sliding_df['dist_bin'] = (sliding_df['distance'] // 16) * 16
profile = sliding_df.groupby('dist_bin')['delta_nll'].agg(['mean', 'std', 'count'])
profile['sem'] = profile['std'] / np.sqrt(profile['count'])

ax.fill_between(profile.index, 
                profile['mean'] - 1.96*profile['sem'],
                profile['mean'] + 1.96*profile['sem'],
                alpha=0.3, color='blue')
ax.plot(profile.index, profile['mean'], 'b-', linewidth=2, label='Mean influence')

ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.axvline(128, color='red', linestyle='--', alpha=0.5, label='STM threshold (128)')

ax.set_xlabel('Distance from Target (tokens)', fontsize=12)
ax.set_ylabel('ΔNLL (influence density)', fontsize=12)
ax.set_title('Continuous Influence vs Distance (Sliding 16-token Window)', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/continuous_influence_curve.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_DIR}/continuous_influence_curve.png")

## Cell 13: Save Results

In [ ]:
# Save all results
results_df.to_csv(f"{OUTPUT_DIR}/doc_level_influence_metrics.csv", index=False)
bin_df.to_csv(f"{OUTPUT_DIR}/bin_level_influences.csv", index=False)
sliding_df.to_csv(f"{OUTPUT_DIR}/sliding_window_influences.csv", index=False)

# Save summary statistics
summary = {
    'n_docs': len(results_df),
    'tail_influence_share_mean': results_df['tail_influence_share'].mean(),
    'tail_influence_share_std': results_df['tail_influence_share'].std(),
    'tail_influence_share_ci_low': results_df['tail_influence_share'].mean() - 1.96*results_df['tail_influence_share'].sem(),
    'tail_influence_share_ci_high': results_df['tail_influence_share'].mean() + 1.96*results_df['tail_influence_share'].sem(),
    'gini_mean': results_df['gini'].mean(),
    'gini_std': results_df['gini'].std(),
    'top1_share_mean': results_df['top1_share'].mean(),
    'top10_share_mean': results_df['top10_share'].mean(),
    'config': {
        'max_context': MAX_CONTEXT,
        'bin_size': BIN_SIZE,
        'burn_in': BURN_IN,
        'target_len': TARGET_LEN,
        'sliding_window': SLIDING_WINDOW,
    }
}

with open(f"{OUTPUT_DIR}/summary.json", 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\nAll results saved to: {OUTPUT_DIR}")
print("\nFiles:")
!ls -la {OUTPUT_DIR}/

## Cell 14: Interpretation

In [ ]:
print("=" * 70)
print("INTERPRETATION: RESIDUAL INFLUENCE vs SPARSE RETRIEVAL")
print("=" * 70)

tail_share = results_df['tail_influence_share'].mean()
gini = results_df['gini'].mean()
top1 = results_df['top1_share'].mean()

print(f"\n1. TAIL INFLUENCE (beyond 128 tokens):")
print(f"   Mean share: {tail_share:.1%}")
if tail_share > 0.1:
    print(f"   → SUPPORTS continuous influence: substantial predictive support from distant context")
else:
    print(f"   → Limited distant influence: most support comes from recent context")

print(f"\n2. BURSTINESS (is influence concentrated or spread?):")
print(f"   Gini coefficient: {gini:.3f} (0=equal, 1=concentrated)")
print(f"   Top-1 bin share: {top1:.1%}")

if gini < 0.4 and top1 < 0.3:
    print(f"   → SUPPORTS residual influence: influence spread across many bins")
    print(f"   → NOT consistent with sparse retrieval bursts")
elif gini > 0.6 or top1 > 0.5:
    print(f"   → SUPPORTS sparse retrieval: influence concentrated in few bins")
else:
    print(f"   → MIXED: moderate concentration of influence")

print(f"\n3. OVERALL CONCLUSION:")
if tail_share > 0.1 and gini < 0.5:
    print(f"   The data SUPPORT the 'memory as residual influence' hypothesis:")
    print(f"   - Predictive support extends beyond STM range")
    print(f"   - Influence is distributed, not bursty")
    print(f"   - This is consistent with continuous, gradient-like memory decay")
else:
    print(f"   The data suggest a more complex picture.")